In [ ]:
import os
import psutil

process = psutil.Process(os.getpid())

print(f"CPU usage: {process.cpu_percent(interval=1)}%")
print(f"RAM usage: {process.memory_info().rss / 1024**2:.2f} MB")
print(f"System RAM: {psutil.virtual_memory().percent}%")
print(f"Disk usage: {psutil.disk_usage('/').percent}%")

In [5]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    filename="app.log",
)

logger = logging.getLogger(__name__)

logger.info("ETL pipeline started")
logger.warning("Data contains missing values")
logger.error("Database connection failed")

In [6]:
import logging
import structlog

logging.basicConfig(
    format="%(message)s",
    level=logging.INFO,
)

structlog.configure(
    processors=[
        structlog.processors.add_log_level,
        structlog.processors.TimeStamper(fmt="iso"),
        structlog.processors.JSONRenderer(),
    ],
    logger_factory=structlog.stdlib.LoggerFactory(),
)

log = structlog.get_logger()

log.info("etl_started", pipeline="sales", rows=10000)
log.error("database_failed", database="postgres", retry=3)

In [7]:
"""prometheus-client exposes application metrics such as request counts, 
execution duration, errors, and resource usage for Prometheus to scrape."""

from fastapi import FastAPI
from prometheus_client import Counter, Histogram, make_asgi_app
import time

app = FastAPI()
app.mount("/metrics", make_asgi_app())

REQUESTS = Counter(
    "api_requests_total",
    "Total API requests",
    ["endpoint", "method", "status"],
)

DURATION = Histogram(
    "api_request_duration_seconds",
    "API request duration",
)

@app.get("/sales")
def get_sales():
    start = time.perf_counter()

    try:
        return {"sales": 1000}
    finally:
        DURATION.observe(time.perf_counter() - start)
        REQUESTS.labels("/sales", "GET", "200").inc()

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider

trace.set_tracer_provider(TracerProvider())
tracer = trace.get_tracer(__name__)

with tracer.start_as_current_span("extract_sales"):
    print("Extracting sales data")